# HunyuanVideo Inference Notebook
This notebook recreates `sample_video.py` so you can run HunyuanVideo inference on Google Colab.\nMake sure you've cloned the repo and prepared model weights before running inference.

In [ ]:
import subprocess, pathlib
repo_url = 'https://github.com/Tencent/HunyuanVideo.git'  # TODO: replace with fork or custom repo if needed
repo_dir = pathlib.Path('/content/HunyuanVideo')
if repo_dir.exists():
    print(f'Repo already exists at {repo_dir}. Skipping clone.')
else:
    print(f'Cloning {repo_url} ...')
    subprocess.check_call(['git', 'clone', repo_url, str(repo_dir)])
    print('Clone complete.')
%cd /content/HunyuanVideo


In [ ]:
import sys, subprocess, pathlib
repo_root = pathlib.Path.cwd()
req_path = repo_root / 'requirements.txt'
if req_path.exists():
    print('Installing dependencies from requirements.txt ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)])
else:
    print('requirements.txt not found, installing core packages ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'loguru', 'einops', 'accelerate', 'diffusers', 'transformers'])
print('Dependency installation complete.')

## Configure model paths
Provide the local path (on Colab) to your prepared HunyuanVideo checkpoints.

In [ ]:
from pathlib import Path
MODEL_BASE = Path('/content/drive/MyDrive/hunyuan/models')  # TODO: update this path
SAVE_PATH = Path('/content/hunyuan_results')
SAVE_PATH.mkdir(parents=True, exist_ok=True)
print('Model base:', MODEL_BASE)
print('Output directory:', SAVE_PATH)

## Helper utilities
We convert a Python dictionary into CLI-style arguments and run the sampler just like `sample_video.py`.

In [ ]:
import math
import time
from datetime import datetime
from loguru import logger
from hyvideo.config import parse_args
from hyvideo.inference import HunyuanVideoSampler
from hyvideo.utils.file_utils import save_videos_grid

def dict_to_args(cfg):
    arg_list = []
    for key, value in cfg.items():
        flag = key.replace('_', '-')
        if isinstance(value, bool):
            if value:
                arg_list.append(f'--{flag}')
        elif isinstance(value, (list, tuple)):
            arg_list.append(f'--{flag}')
            arg_list.extend([str(v) for v in value])
        elif value is not None:
            arg_list.append(f'--{flag}')
            arg_list.append(str(value))
    return parse_args(arg_list)

def run_hunyuan_inference(config):
    args = dict_to_args(config)
    if not args.model_base.exists():
        raise ValueError(f'Model base not found: {args.model_base}')
    logger.info('Loading sampler ...')
    sampler = HunyuanVideoSampler.from_pretrained(args.model_base, args=args)
    args = sampler.args
    logger.info('Running inference ...')
    outputs = sampler.predict(
        prompt=args.prompt,
        height=args.video_size[0],
        width=args.video_size[1],
        video_length=args.video_length,
        seed=args.seed,
        negative_prompt=args.neg_prompt,
        infer_steps=args.infer_steps,
        guidance_scale=args.cfg_scale,
        num_videos_per_prompt=args.num_videos,
        flow_shift=args.flow_shift,
        batch_size=args.batch_size,
        embedded_guidance_scale=args.embedded_guidance_scale,
    )
    samples = outputs['samples']
    saved_paths = []
    for i, sample in enumerate(samples):
        sample = sample.unsqueeze(0)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"{timestamp}_seed{outputs['seeds'][i]}_{outputs['prompts'][i][:60].replace('/', '')}.mp4"
        out_path = SAVE_PATH / filename
        save_videos_grid(sample, str(out_path), fps=24)
        logger.info(f'Saved video: {out_path}')
        saved_paths.append(out_path)
    return saved_paths, outputs

def show_videos(video_paths):
    from IPython.display import Video, display
    for path in video_paths:
        display(Video(filename=str(path), embed=True))


## Configure inference parameters

In [ ]:
default_config = {
    'model_base': MODEL_BASE,
    'save_path': SAVE_PATH,
    'prompt': 'A cat walks on the grass, realistic style.',
    'neg_prompt': '',
    'video_size': [544, 960],
    'video_length': 129,
    'infer_steps': 30,
    'seed': 42,
    'num_videos': 1,
    'batch_size': 1,
    'flow_shift': 5.0,
    'cfg_scale': 6.0,
    'layer_offload': True,
    'offload_blocks': '0,1,2,3,4,20,21',
    'int8_cache_offload': True,
    'int8_quant_mode': 'channel',
    'sparsity_enable': True,
    'sparsity_attn_threshold': 5e-4,
    'sparsity_mlp_threshold': 1e-4,
    'sparsity_head_target': 0.6,
    'sparsity_mlp_target': 0.5,
    'sparsity_warmup_steps': 8,
    'sparsity_decay': 0.9,
    'sparsity_min_heads': 1,
    'sparsity_min_mlp': 32,
    'temporal_cache_enable': True,
    'temporal_cache_threshold': 1e-4,
    'temporal_cache_max_steps': 6,
    'temporal_cache_mode': 'blend',
    'temporal_cache_rank': 12,
    'temporal_cache_proj_dim': 48,
}
default_config

## Run inference

In [ ]:
video_paths, outputs = run_hunyuan_inference(default_config)
print('Seeds:', outputs['seeds'])
show_videos(video_paths)